In [ ]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"]="0"
import torch

# import os
# os.environ["CUDA_VISIBLE_DEVICES"]="1"
torch.cuda.set_device(0)


import numpy as np
import tensorflow as tf
import pandas as pd
import pyarabic.araby as araby
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.initializers import TruncatedNormal
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import CategoricalAccuracy
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, Dataset, concatenate_datasets
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 1000)


log_file = 'log_raw_13.txt'
with open(log_file, 'w') as f:
    f.write('Model,Accuracy,F1\n')



df = pd.read_json("RE_dataset_Binary_multi_without_F_1.json")

# remove_labels = ["R", "INT", "QU"]
remove_labels = ["SA", "A", "FT", "R", "INT", "QU"]   # less than 140 samples
df = df[~df["label"].isin(remove_labels)]
df = df.reset_index(drop=True)


df.fillna('', inplace=True)

display(len(df))
display(df[:4])



df = df[df['text'] != '']

display(len(df))

classes = set(df['label'].values)
display(classes)

# return

df['label'] = df['label'].astype('category')
df['label'] = df['label'].cat.codes



df = df[['text', 'label']]


classes_num = len(classes)
display(classes_num)
display(len(df))


ds = Dataset.from_pandas(df)

ds = ds.train_test_split(test_size=0.2, seed=42)
display(ds)

max_sequence_length = 128


models = [


    'microsoft/deberta-v3-base',
    'FacebookAI/roberta-base',
    'bert-base-cased',
]


for model_name in models:
    for i in range(3):
        print(f'{model_name}, try:{i}')

        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name,
                                                              num_labels=classes_num).to('cuda')
        dataset_train = ds['train']
        dataset_validation = ds['test']



        def preprocess_function(examples):
            return tokenizer(examples['text'], truncation=True, padding="max_length",
                            max_length=max_sequence_length, add_special_tokens=True)


        dataset_train = dataset_train.map(preprocess_function, batched=True)
        dataset_validation = dataset_validation.map(preprocess_function, batched=True)



        def compute_metrics(eval_pred):
            logits, labels = eval_pred
            predictions = np.argmax(logits, axis=-1)
            acc = accuracy_score(labels, predictions)
            f1 = f1_score(labels, predictions, average='micro')
            with open(log_file, 'a') as f:
                f.write(f'{model_name},{acc},{f1}\n')
            return {'accuracy': acc, 'f1_score': f1}


        epochs = 25
        save_steps = 10000 #save checkpoint every 10000 steps
        batch_size = 32

        training_args = TrainingArguments(
            output_dir = 'bert/',
            overwrite_output_dir=True,
            num_train_epochs = epochs,
            per_device_train_batch_size = batch_size,
            per_device_eval_batch_size = batch_size,
            save_steps = save_steps,
            save_total_limit = 1, #only save the last 5 checkpoints
            fp16=True,
            learning_rate = 5e-5,  # 5e-5 is the default
            logging_steps = 50, #50_000
            evaluation_strategy = 'steps',
            # evaluate_during_training = True,
            eval_steps = 50

        )

        trainer = Trainer(
            model = model,
            args = training_args,
            # data_collator=data_collator,
            train_dataset=dataset_train,
            eval_dataset=dataset_validation,
            compute_metrics = compute_metrics
        )


        # trainer.train(resume_from_checkpoint=True)
        trainer.train()


results = pd.read_csv(log_file)

best_results = results.groupby('Model', as_index=False)['F1'].max()

best_results = pd.merge(best_results, results, on=['Model', 'F1'])
best_results = best_results[['Model', 'Accuracy', 'F1']]
best_results = best_results.drop_duplicates()
best_results.to_csv('results_3_13.csv')
display(best_results)



2025-09-20 00:29:05.616655: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-20 00:29:05.639646: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-20 00:29:06.163121: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


1799

,label,text
0,CO,The Libra scheduler will be a sub-component of SGE.
1,O,All coding will be done in standard C
2,CO,Exhaustive searches of the entire set of combinations of jobs will not be done. Heuristics will be developed for this scheduling problem.
3,CO,The Clarus system shall be able to be hosted at one or more physical locations.


1799

{'CO', 'MN', 'O', 'PE', 'PO', 'SE', 'US'}

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/1439 [00:00<?, ? examples/s]

Map:   0%|          | 0/360 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Accuracy,F1 Score
50,1.338500,0.891059,0.713889,0.713889
100,0.436000,0.652106,0.794444,0.794444
150,0.117600,0.626819,0.813889,0.813889
200,0.035600,0.774282,0.827778,0.827778
250,0.017200,1.145201,0.780556,0.780556
300,0.005200,1.236201,0.847222,0.847222
350,0.005200,1.552450,0.825000,0.825000
400,0.002900,1.700754,0.844444,0.844444
450,0.000000,1.897050,0.838889,0.838889
500,0.000000,1.945926,0.844444,0.844444


,Model,Accuracy,F1
0,FacebookAI/roberta-base,0.819444,0.819444
3,bert-base-cased,0.850000,0.850000
6,microsoft/deberta-v3-base,0.802778,0.802778


In [ ]:
import pandas as pd

df = pd.read_json("RE_dataset_Binary_multi_without_F_1.json")

counts = df["label"].value_counts()

print(counts)

label
US     515
SE     362
MN     221
PE     215
PO     173
CO     172
O      141
SA     126
A      118
FT     101
R       72
INT     58
QU      39
Name: count, dtype: int64
